# S5 — EDA 大統整：從零到報告（Solution）

> **時間**：2 小時（講授 70min + 課堂練習 40min + QA 10min）  
> **資料**：`datasets/spaceship-titanic/train.csv`  
> **學完能幹嘛**：面對任何 Kaggle 資料集，用系統化的 EDA 流程產出完整分析報告

**一句話口訣：比賽前 80% 的時間在看資料，只有 20% 在調模型。**

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

df = pd.read_csv('datasets/spaceship-titanic/train.csv')

---
## Step 1-4: (同 student 版，此處省略重複)

請參考 student 版的 Step 1-4 完整內容。以下直接進入練習解答。

In [ ]:
# 資料準備（與 student 版相同）
df_clean = df.copy()
df_clean['Transported_int'] = df_clean['Transported'].astype(float)

spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
for col in spend_cols + ['Age']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())
for col in ['HomePlanet', 'Destination', 'CryoSleep', 'VIP']:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

df_clean['TotalSpending'] = df_clean[spend_cols].sum(axis=1)
cabin_split = df_clean['Cabin'].str.split('/', expand=True)
df_clean['Cabin_deck'] = cabin_split[0]
df_clean['Cabin_num'] = pd.to_numeric(cabin_split[1], errors='coerce')
df_clean['Cabin_side'] = cabin_split[2]
df_clean['NoSpending'] = (df_clean['TotalSpending'] == 0).astype(int)
df_clean['Group'] = df_clean['PassengerId'].str.split('_').str[0]
df_clean['GroupSize'] = df_clean.groupby('Group')['Group'].transform('count')
df_clean['IsAlone'] = (df_clean['GroupSize'] == 1).astype(int)

# 相關性
numeric_df = df_clean.select_dtypes(include='number')
corr_with_target = numeric_df.corr()['Transported_int'].drop('Transported_int').sort_values()

print('✅ 資料準備完成')

---
## 課堂練習 — Solution

### 🟢 送分題

In [ ]:
# 🟢 送分題 Solution
corr_cols = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
corr_matrix = df_clean[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('數值特徵相關性矩陣', fontsize=14)
plt.tight_layout()
plt.show()

### 🟡 核心題

In [ ]:
# 🟡 核心題 Solution — 2×3 Dashboard
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Transported 圓餅圖
df_clean['Transported'].value_counts().plot.pie(
    autopct='%1.1f%%', colors=['#ff9999', '#66b2ff'], ax=axes[0, 0])
axes[0, 0].set_title('Transported Distribution')
axes[0, 0].set_ylabel('')

# 2. Age KDE by Transported
for val in [True, False]:
    subset = df_clean[df_clean['Transported'] == val]['Age'].dropna()
    sns.kdeplot(subset, fill=True, ax=axes[0, 1], label=f'{val}', alpha=0.5)
axes[0, 1].set_title('Age KDE by Transported')
axes[0, 1].legend()

# 3. CryoSleep
df_clean.groupby('CryoSleep')['Transported_int'].mean().plot(
    kind='bar', color=['#ff9999', '#66b2ff'], ax=axes[0, 2])
axes[0, 2].set_title('Transported by CryoSleep')
axes[0, 2].set_xticklabels(['False', 'True'], rotation=0)

# 4. HomePlanet
df_clean.groupby('HomePlanet')['Transported_int'].mean().plot(
    kind='bar', color='steelblue', ax=axes[1, 0])
axes[1, 0].set_title('Transported by HomePlanet')
axes[1, 0].set_xticklabels(axes[1, 0].get_xticklabels(), rotation=0)

# 5. Cabin_deck
deck_rate = df_clean.groupby('Cabin_deck')['Transported_int'].mean().sort_values(ascending=False)
deck_rate.plot(kind='bar', color='steelblue', ax=axes[1, 1])
axes[1, 1].set_title('Transported by Cabin_deck')

# 6. TotalSpending boxplot
df_clean.boxplot(column='TotalSpending', by='Transported', ax=axes[1, 2])
axes[1, 2].set_title('TotalSpending by Transported')
axes[1, 2].set_xlabel('Transported')
plt.suptitle('')  # 移除自動標題

plt.suptitle('Spaceship Titanic — EDA Dashboard', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

### 🔴 挑戰題

這是一個 take-home 作業，建議新開一個 notebook 完成 House Prices 的完整 EDA。

以下提供框架作為參考：

In [ ]:
# 🔴 挑戰題 Solution 框架
hp = pd.read_csv('datasets/house-prices/train.csv')

# === Step 1: First Look ===
print(f'Shape: {hp.shape}')
print(f'Target: SalePrice')
print(f'  mean={hp["SalePrice"].mean():,.0f}, median={hp["SalePrice"].median():,.0f}')
print(f'  skew={hp["SalePrice"].skew():.2f} → 右偏，考慮 log 轉換')

# === Step 2: Missing Data ===
na_info = hp.isnull().sum().sort_values(ascending=False)
na_info = na_info[na_info > 0]
print(f'\n有缺值的欄位: {len(na_info)} 個')
print(na_info.head(5))

# === Step 3: Distribution ===
top_skew = hp.select_dtypes(include='number').skew().sort_values(ascending=False).head(5)
print(f'\n最偏態的 5 個欄位:')
print(top_skew)

# === Step 4: Feature Engineering ===
hp['TotalSF'] = hp['TotalBsmtSF'] + hp['1stFlrSF'] + hp['2ndFlrSF']
hp['TotalBath'] = hp['FullBath'] + 0.5 * hp['HalfBath'] + hp['BsmtFullBath'] + 0.5 * hp['BsmtHalfBath']
hp['HasPool'] = (hp['PoolArea'] > 0).astype(int)
print(f'\n新特徵: TotalSF, TotalBath, HasPool')

# === Step 5: Correlation ===
corr_sale = hp.select_dtypes(include='number').corr()['SalePrice'].sort_values(ascending=False)
print(f'\n與 SalePrice 相關性前 10：')
print(corr_sale.head(11))  # 包含自己

In [ ]:
# Step 6: Summary Dashboard
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. SalePrice 分布
sns.histplot(hp['SalePrice'], bins=30, kde=True, ax=axes[0, 0], color='steelblue')
axes[0, 0].set_title('SalePrice Distribution')

# 2. SalePrice log
sns.histplot(np.log1p(hp['SalePrice']), bins=30, kde=True, ax=axes[0, 1], color='coral')
axes[0, 1].set_title('log(SalePrice) Distribution')

# 3. 缺值 top 10
na_top10 = (hp.isnull().mean() * 100).sort_values(ascending=False).head(10)
na_top10.plot(kind='barh', color='coral', ax=axes[0, 2])
axes[0, 2].set_title('Top 10 Missing (%)')

# 4. Top 相關特徵
corr_top = corr_sale.drop('SalePrice').head(8)
corr_top.plot(kind='bar', color='steelblue', ax=axes[1, 0])
axes[1, 0].set_title('Top 8 Correlated Features')
axes[1, 0].tick_params(axis='x', rotation=45)

# 5. OverallQual vs SalePrice
sns.boxplot(x='OverallQual', y='SalePrice', data=hp, ax=axes[1, 1])
axes[1, 1].set_title('SalePrice by OverallQual')

# 6. GrLivArea vs SalePrice
axes[1, 2].scatter(hp['GrLivArea'], hp['SalePrice'], alpha=0.5, s=15)
axes[1, 2].set_title('GrLivArea vs SalePrice')
axes[1, 2].set_xlabel('GrLivArea')
axes[1, 2].set_ylabel('SalePrice')

plt.suptitle('House Prices — EDA Summary Dashboard', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

---
## 全課程速查

| Session | 核心口訣 |
|:--------|:---------|
| S1 | 拿到資料的前 10 分鐘，決定接下來 10 小時的方向 |
| S2 | 缺值有三種死法：MCAR、MAR、MNAR |
| S3 | 偏態是脾氣，離群值是秘密 |
| S4 | 原始欄位是食材，特徵工程是料理 |
| S5 | 80% 時間看資料，20% 調模型 |